# ╔══════════════════════════════════════════════════════════════════╗
# ║  Geosteering AI v2.0 — Notebook Integrador                    ║
# ║  Pipeline End-to-End: Dados → Modelo → Treinamento → Inferência║
# ╚══════════════════════════════════════════════════════════════════╝

## Geosteering AI v2.0 — Notebook Integrador Principal

**Autor:** Daniel Leal  
**Framework:** TensorFlow 2.x / Keras (exclusivo — PyTorch PROIBIDO)  
**Ambiente:** Google Colab Pro+ GPU (T4/A100)  
**Versão:** v2.0.2 (2026-03 — Fase E, notebook integrador)

---

### O Problema Físico

Este notebook demonstra o pipeline completo de **inversão 1D de resistividade elétrica** 
via Deep Learning para **geosteering em tempo real**.

```
Ferramenta LWD (20 kHz)                    Rede Neural Profunda
┌─────────────────────┐                   ┌──────────────────────┐
│  Tensor EM H (3×3)  │   ──────────►     │  (batch, 600, 5)     │
│  Re + Im = 18 vals  │   Inversão        │       ↓              │
│  a cada ~5 segundos │   Deep Learning   │  (batch, 600, 2)     │
└─────────────────────┘                   │  [ρ_h, ρ_v] Ohm.m   │
       Medidas EM                         └──────────────────────┘
                                               Resistividades
```

**Fluxo completo do notebook:**
1. 🔧 Instalação e configuração de GPU
2. ⚙️ Configuração do pipeline (PipelineConfig)
3. 📊 Carregamento e preparação de dados
4. 🏗️ Construção do modelo (ResNet-18 default)
5. 🎯 Compilação e callbacks
6. 🚂 Treinamento com curriculum noise
7. 📈 Avaliação e métricas
8. 🎨 Visualização dos resultados
9. 💾 Salvamento do InferencePipeline
10. 🚀 Smoke tests GPU (forward + backward)
11. 🧪 Smoke tests com dados sintéticos

## Célula 0 — Instalação de Dependências

Instala `geosteering-ai` diretamente do repositório GitHub.  
Em Colab: executar uma única vez (reiniciar runtime após instalação se solicitado).

In [ ]:
# ── Instalação do pacote geosteering-ai ──────────────────────────────────
# Em Colab Pro+, instalar via GitHub para obter a versão mais recente.
# Alternativa: !pip install geosteering-ai (quando publicado no PyPI)
#
# Nota: Se instalar localmente (VSCode), o pacote já está disponível
# via `pip install -e .` no diretório raiz do projeto.
import sys

# Verifica se já está instalado (ambiente local)
try:
    import geosteering_ai
    print(f"✓ geosteering_ai já disponível: v{geosteering_ai.__version__}")
except ImportError:
    # Instala do GitHub (Colab)
    print("Instalando geosteering-ai do GitHub...")
    !pip install -q git+https://github.com/daniel-leal/geosteering-ai.git@main
    print("✓ Instalação concluída. Reinicie o runtime se solicitado.")

# Verifica versão final
import geosteering_ai
print(f"geosteering_ai v{geosteering_ai.__version__} — {geosteering_ai.__framework__}")

## Célula 1 — Configuração de GPU

Configura TensorFlow para uso eficiente da GPU disponível no Colab:  
- `memory_growth=True`: alocação dinâmica de VRAM (evita OOM em Colab Pro+)
- XLA opcional para fusão de operações e ganho de throughput de 20-40%

In [ ]:
# ── Configuração de GPU ──────────────────────────────────────────────────
# setup_gpu() habilita memory_growth para todos os dispositivos GPU.
# Retorna dict com: gpu_count, memory_growth_set, devices.
import logging
import os

# Suprimir warnings verbosos do TensorFlow antes do import
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

from geosteering_ai.utils import setup_gpu, get_logger

# Configura logger do pipeline (nível INFO para produção, DEBUG para debug)
logger = get_logger("main", level=logging.INFO)

# Configura GPU — memory_growth=True para Colab Pro+ (evita OOM)
gpu_info = setup_gpu(memory_growth=True)
print(f"GPUs disponíveis: {gpu_info['gpu_count']}")
print(f"Memory growth ativo: {gpu_info['memory_growth_set']}")
for dev in gpu_info.get('devices', []):
    print(f"  → {dev}")

# Verifica TensorFlow
import tensorflow as tf
print(f"\nTensorFlow: {tf.__version__}")
print(f"GPU disponível (TF): {tf.test.is_gpu_available() if hasattr(tf.test, 'is_gpu_available') else bool(tf.config.list_physical_devices('GPU'))}")
print(f"Dispositivos físicos: {[d.name for d in tf.config.list_physical_devices()]}")

## Célula 2 — Configuração do Pipeline (PipelineConfig)

`PipelineConfig` é o único ponto de verdade — toda função recebe `config` como parâmetro.  
Presets disponíveis: `baseline()`, `robusto()`, `nstage(n)`, `geosinais_p4()`, `realtime()`.

In [ ]:
# ── PipelineConfig — Ponto único de verdade ──────────────────────────────
# Usar preset PipelineConfig.robusto() como ponto de partida.
# Modifique campos específicos com dataclasses.replace() para imutabilidade.
#
# Campos críticos validados pelo __post_init__ (Errata v4.4.5):
#   frequency_hz   = 20000.0  (default, range [100, 1e6] Hz)
#   spacing_meters = 1.0      (default, range [0.1, 10.0] m)
#   sequence_length = 600     (default, range [10, 100000])
#   target_scaling = "log10"  (NUNCA "log")

import dataclasses
from geosteering_ai.config import PipelineConfig

# ── Opção A: Preset E-Robusto (recomendado para início) ─────────────────
config = PipelineConfig.robusto()

# ── Opção B: Configuração personalizada ──────────────────────────────────
# config = dataclasses.replace(
#     PipelineConfig.robusto(),
#     model_type="ResNet_18",       # Arquitetura default
#     epochs=50,                    # Reduzir para smoke test
#     early_stopping_patience=15,   # Paciência menor para testes
#     use_tensorboard=True,         # Habilitar TensorBoard no Colab
#     use_gradient_monitor=True,    # Monitorar gradientes reais (Fase E)
#     experiment_tag="smoke_test_v1",
# )

# ── Opção C: Para validação rápida (smoke test) ──────────────────────────
config_smoke = dataclasses.replace(
    PipelineConfig.baseline(),
    epochs=5,
    early_stopping_patience=5,
    batch_size=8,
    use_noise=False,
    use_curriculum=False,
    use_tensorboard=False,
    use_csv_logger=False,
    experiment_tag="smoke_test",
)

print("=" * 60)
print(f"Config: PipelineConfig.robusto()")
print(f"  model_type       : {config.model_type}")
print(f"  sequence_length  : {config.sequence_length}")
print(f"  input_features   : {config.input_features}")
print(f"  output_targets   : {config.output_targets}")
print(f"  target_scaling   : {config.target_scaling}")
print(f"  frequency_hz     : {config.frequency_hz}")
print(f"  spacing_meters   : {config.spacing_meters}")
print(f"  learning_rate    : {config.learning_rate}")
print(f"  epochs           : {config.epochs}")
print(f"  use_noise        : {config.use_noise}")
print(f"  noise_level_max  : {config.noise_level_max}")
print("=" * 60)
print(f"\nConfig smoke test (5 épocas):")
print(f"  epochs           : {config_smoke.epochs}")
print(f"  batch_size       : {config_smoke.batch_size}")
print(f"  use_noise        : {config_smoke.use_noise}")

## Célula 3 — Carregamento e Preparação de Dados

`DataPipeline.prepare(dat_path, out_path)` executa a cadeia completa:  
`.dat` → parse → split-by-model [P1] → fit_scaler (dados limpos) → `PreparedData`

**Cadeia fisicamente correta:**
```
train_raw → noise(A/m) → FV_tf(noisy) → GS_tf(noisy) → scale → modelo
                                │
                      GS veem ruído ✓ (fidelidade LWD)
```

In [ ]:
# ── DataPipeline — Preparação de Dados ─────────────────────────────────
# Substitua os caminhos abaixo pelos seus arquivos .dat e .out.
# Formato .dat: binário Fortran, 22 colunas (ver docs/physics/onboarding.md)
# Formato .out: metadados do modelo geológico (cabeçalho ASCII Fortran)
#
# Colunas do .dat (22-col):
#   Col 1: zobs (profundidade)  → INPUT_FEATURE
#   Col 2: res_h (ρ_h Ohm.m)   → OUTPUT_TARGET
#   Col 3: res_v (ρ_v Ohm.m)   → OUTPUT_TARGET
#   Col 4-5: Re/Im(Hxx) → planar EM → INPUT_FEATURE
#   Col 20-21: Re/Im(Hzz) → axial EM → INPUT_FEATURE

from geosteering_ai.data import DataPipeline

# ── Configurar caminhos (Google Drive — Colab Pro+) ─────────────────────
# Passo 1: Montar Google Drive no Colab
# from google.colab import drive
# drive.mount('/content/drive')
#
# Passo 2: Caminhos reais do dataset Inv0Dip13mil (13.000 modelos 1D)
#   Localização: Google Drive → DataSets/Resistivity_Projeto_Pos_Doutorado/
_DRIVE_BASE = "/content/drive/MyDrive/DataSets/Resistivity_Projeto_Pos_Doutorado"

# Dataset principal (treino + validação + teste)
DAT_PATH = f"{_DRIVE_BASE}/Inv0Dip13mil.dat"       # Binário Fortran 22-col
OUT_PATH = f"{_DRIVE_BASE}/Inv0Dip13mil.out"        # Metadados do modelo geológico

# Dataset holdout (avaliação independente)
HOLDOUT_DAT_PATH = f"{_DRIVE_BASE}/Inv0Dip13mil_holdout.dat"
HOLDOUT_OUT_PATH = f"{_DRIVE_BASE}/Inv0Dip13mil_holdout.out"

# ── MODO SINTÉTICO (para smoke test sem dados reais) ────────────────────
# Se não tiver dados reais, use dados sintéticos para validação do pipeline.
# Ver Célula 10 (Smoke Tests) para geração sintética completa.
USE_SYNTHETIC_DATA = True   # ← Mude para False quando tiver dados reais no Drive

if USE_SYNTHETIC_DATA:
    print("MODO SINTÉTICO: usando dados gerados numericamente.")
    print("Configure USE_SYNTHETIC_DATA=False quando tiver dados .dat/.out reais.")
    print(f"  Esperado: {DAT_PATH}")
    prepared_data = None   # Será gerado na Célula 10
else:
    import os
    if not os.path.exists(DAT_PATH):
        raise FileNotFoundError(
            f"Arquivo não encontrado: {DAT_PATH}\n"
            "Monte o Google Drive primeiro: drive.mount('/content/drive')"
        )
    print(f"Carregando dados: {DAT_PATH}")
    pipeline = DataPipeline(config)
    prepared_data = pipeline.prepare(DAT_PATH, OUT_PATH)
    
    print(f"\n✓ PreparedData carregado:")
    print(f"  train: {prepared_data.x_train.shape} → {prepared_data.y_train.shape}")
    print(f"  val  : {prepared_data.x_val.shape} → {prepared_data.y_val.shape}")
    print(f"  test : {prepared_data.x_test.shape} → {prepared_data.y_test.shape}")
    print(f"  n_models_train: {prepared_data.n_models_train}")
    print(f"  n_models_val  : {prepared_data.n_models_val}")
    print(f"  scaler_params : {list(prepared_data.scaler_params.keys())}")

## Célula 4 — Construção do Modelo

`build_model(config)` usa `ModelRegistry` para instanciar uma das 44 arquiteturas disponíveis.  
Default: **ResNet-18★** (Tier 1, causal-compatible, melhor tradeoff performance/velocidade).

**Output shape:** `(batch, 600, 2)` — seq2seq preservando dimensão temporal.

```
Input  (batch, None, N_FEATURES)    →  N_FEATURES = 5 (baseline P1)
                ↓
   ResNet-18: Stem → 4×Stage(ResBlock) → Output Head
                ↓
Output (batch, None, output_channels)  →  output_channels = 2 [ρ_h, ρ_v]
```

In [ ]:
# ── ModelRegistry — Construção do Modelo ─────────────────────────────────
# build_model(config) despacha para a arquitetura especificada em
# config.model_type. Todas as 44 arquiteturas implementam a mesma
# interface: Input(batch, None, N_FEATURES) → Output(batch, None, 2).

from geosteering_ai.models import build_model, list_available_models, get_model_info

# Listar arquiteturas disponíveis (opcional)
available = list_available_models()
print(f"Arquiteturas disponíveis: {len(available)} modelos em 9 famílias")
print(f"  Família CNN: {[m for m in available if 'ResNet' in m or 'CNN' in m or 'Conv' in m or 'Inception' in m][:6]}")
print(f"  Família GS : {[m for m in available if m in ['WaveNet','Causal_Transformer','Mamba_S4','Informer','Encoder_Forecaster']]}")

# Usar config_smoke para smoke test rápido
# Para treinamento real: use `config` (PipelineConfig.robusto())
active_config = config_smoke

# Constrói o modelo
model = build_model(active_config)

# Info da arquitetura
info = get_model_info(active_config.model_type)
print(f"\n{'='*60}")
print(f"Modelo: {active_config.model_type}")
print(f"  Família  : {info.get('family', 'N/A')}")
print(f"  Tier     : {info.get('tier', 'N/A')}")
print(f"  Causal   : {info.get('causal_compatible', 'N/A')}")
print(f"  Parâmetros: {model.count_params():,}")

# Verificar shape de saída com forward pass dummy
import numpy as np
import tensorflow as tf

n_feat = len(active_config.input_features)
x_dummy = tf.random.normal([2, active_config.sequence_length, n_feat])
y_dummy_pred = model(x_dummy, training=False)
print(f"\nSmoke test shape:")
print(f"  Input  : {x_dummy.shape}")
print(f"  Output : {y_dummy_pred.shape}")
assert y_dummy_pred.shape == (2, active_config.sequence_length, 2), \
    f"Shape inválido: {y_dummy_pred.shape}"
print(f"  ✓ Shape correto: (batch, {active_config.sequence_length}, 2) [ρ_h, ρ_v]")

# Summary do modelo
model.summary(line_length=80)

## Célula 5 — Loss Function e Metrics

`LossFactory.get(config)` seleciona a loss function baseada em `config.loss_type`.  
`build_metrics(config)` constrói as métricas: R², PerComponentMetric, AnisotropyRatioError.

In [ ]:
# ── LossFactory + Metrics ────────────────────────────────────────────────
# 26 loss functions disponíveis (13 genéricas + 4 geofísicas + 2 geosteering
# + 6 avançadas + 1 híbrida PINNs).
# Default: log_scale_aware (geofísica — aware da escala log10 de ρ).

from geosteering_ai.losses import LossFactory, list_available_losses
from geosteering_ai.training import build_metrics

# Listar losses disponíveis
losses = list_available_losses()
print(f"Losses disponíveis: {len(losses)}")
print(f"  Ativas: {active_config.loss_type}")

# Construir loss function
loss_fn = LossFactory.get(active_config)
loss_name = getattr(loss_fn, "__name__", type(loss_fn).__name__)
print(f"\nLoss function: {loss_name}")

# Construir métricas
metrics_list = build_metrics(active_config)
print(f"Métricas: {[type(m).__name__ for m in metrics_list]}")

# Verificar loss com dados sintéticos
y_true_dummy = tf.random.normal([2, active_config.sequence_length, 2])
y_pred_dummy = tf.random.normal([2, active_config.sequence_length, 2])
loss_val = loss_fn(y_true_dummy, y_pred_dummy)
print(f"\n✓ Loss smoke test: {float(loss_val):.6f} (dtype: {loss_val.dtype})")

## Célula 6 — Callbacks e TrainingLoop

`build_callbacks(config)` monta automaticamente a lista de callbacks baseada no config.  
Callbacks obrigatórios: `EarlyStopping`, `BestEpochTracker`, `EpochSummary`.  
Opcionais: `TensorBoard`, `CSVLogger`, `UpdateNoiseLevelCallback`, `GradientMonitor` (Fase E).

In [ ]:
# ── Build Callbacks ──────────────────────────────────────────────────────
# build_callbacks() constrói a lista de callbacks Keras baseada no config.
# O noise_level_var (tf.Variable) é compartilhado com DataPipeline para
# controle do nível de ruído on-the-fly durante o curriculum learning.

from geosteering_ai.training import build_callbacks, add_gradient_monitor
from geosteering_ai.noise import create_noise_level_var

# Variável de ruído compartilhada (curriculum on-the-fly)
noise_level_var = create_noise_level_var(initial_value=0.0)

# Construir lista de callbacks
callbacks = build_callbacks(
    config=active_config,
    model=model,
    noise_level_var=noise_level_var,
)

print(f"Callbacks montados: {len(callbacks)}")
for i, cb in enumerate(callbacks, 1):
    print(f"  {i:2d}. {type(cb).__name__}")

# ── GradientMonitor (Fase E — opt-in) ────────────────────────────────────
# Adicionar GradientMonitor separadamente (requer sample_batch e loss_fn).
# Só ativo se config.use_gradient_monitor=True.
if active_config.use_gradient_monitor:
    # Criar batch de amostra para o GradientMonitor
    n_feat = len(active_config.input_features)
    x_gm = np.random.randn(8, active_config.sequence_length, n_feat).astype('float32')
    y_gm = np.random.randn(8, active_config.sequence_length, 2).astype('float32')
    callbacks = add_gradient_monitor(
        callbacks, model, loss_fn, (x_gm, y_gm), active_config
    )
    print(f"\n✓ GradientMonitor adicionado (freq={active_config.gradient_monitor_freq} épocas)")
else:
    print(f"\n  GradientMonitor: desativado (use_gradient_monitor=False)")

## Célula 7 — Treinamento (Smoke Test com Dados Sintéticos)

Valida o pipeline end-to-end com dados sintéticos antes de usar dados reais.  
Confirma: forward pass → loss → backward → atualização de pesos → callbacks.

In [ ]:
# ── Smoke Test: Treinamento com Dados Sintéticos ─────────────────────────
# Gera dados sintéticos no formato correto do pipeline:
#   X: (n_models, sequence_length, n_features) — EM + zobs
#   Y: (n_models, sequence_length, 2) — [ρ_h, ρ_v] em escala log10
#
# Este smoke test valida:
#   1. Forward pass: X → model → Y_pred (shapes corretos)
#   2. Loss computation: loss_fn(Y_true, Y_pred) → scalar
#   3. Backward pass: gradients computados sem NaN/Inf
#   4. Weight update: otimizador atualiza pesos
#   5. Callbacks: EarlyStopping, BestEpochTracker, EpochSummary

import numpy as np
import tensorflow as tf
from geosteering_ai.training import TrainingLoop

# Parâmetros do smoke test
N_TRAIN_MODELS = 20   # Modelos geológicos no treino (split P1)
N_VAL_MODELS   = 5    # Modelos geológicos na validação
SEQ_LEN        = active_config.sequence_length   # 600
N_FEAT         = len(active_config.input_features)  # 5

print(f"Gerando dados sintéticos:")
print(f"  Train: {N_TRAIN_MODELS} modelos × {SEQ_LEN} pts × {N_FEAT} features")
print(f"  Val  : {N_VAL_MODELS} modelos × {SEQ_LEN} pts × {N_FEAT} features")

# ── Gera X (features EM + zobs) ──────────────────────────────────────────
# Colunas [1,4,5,20,21]: zobs, Re(Hxx), Im(Hxx), Re(Hzz), Im(Hzz)
# Hxx tipicamente ≈ 1e-6 A/m, Hzz ≈ 1e-6 A/m (após decoupling)
rng = np.random.default_rng(42)
x_train = rng.standard_normal((N_TRAIN_MODELS, SEQ_LEN, N_FEAT)).astype(np.float32)
x_val   = rng.standard_normal((N_VAL_MODELS,   SEQ_LEN, N_FEAT)).astype(np.float32)

# Simula zobs: profundidade crescente [0, SEQ_LEN-1] metros
z = np.linspace(0, SEQ_LEN - 1, SEQ_LEN, dtype=np.float32)
x_train[:, :, 0] = z[np.newaxis, :]   # Col 0 (pos 0 no array) = zobs
x_val[:, :, 0]   = z[np.newaxis, :]

# ── Gera Y (resistividades em escala log10) ──────────────────────────────
# ρ_h e ρ_v em log10: log10(1) = 0, log10(100) ≈ 2, log10(1000) ≈ 3
# Range físico completo: 0.1–10000 Ohm.m → log10: [-1.0, 4.0]
# Smoke test usa subrange representativo: 0.3–1000 Ohm.m → [-0.5, 3.0]
rho_h_log = rng.uniform(-0.5, 3.0, (N_TRAIN_MODELS, SEQ_LEN, 1)).astype(np.float32)
rho_v_log = rho_h_log + rng.uniform(0.0, 0.7, rho_h_log.shape).astype(np.float32)
y_train = np.concatenate([rho_h_log, rho_v_log], axis=-1)   # (N, 600, 2)

rho_h_log_v = rng.uniform(-0.5, 3.0, (N_VAL_MODELS, SEQ_LEN, 1)).astype(np.float32)
rho_v_log_v = rho_h_log_v + rng.uniform(0.0, 0.7, rho_h_log_v.shape).astype(np.float32)
y_val = np.concatenate([rho_h_log_v, rho_v_log_v], axis=-1)

# ── Cria tf.data.Dataset ─────────────────────────────────────────────────
BATCH_SIZE = active_config.batch_size
AUTOTUNE   = tf.data.AUTOTUNE

train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .shuffle(N_TRAIN_MODELS, seed=42)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)
val_ds = (
    tf.data.Dataset.from_tensor_slices((x_val, y_val))
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print(f"\nDatasets criados:")
print(f"  train_ds: {len(list(train_ds))} batches")
print(f"  val_ds  : {len(list(val_ds))} batches")

# ── TrainingLoop.run() ──────────────────────────────────────────────────
# Orquestrador de compile → fit → (opcional) causal_finetune.
# Retorna TrainingResult com: history, best_epoch, best_val_loss, etc.
loop = TrainingLoop(active_config)
print(f"\nIniciando treinamento (smoke test: {active_config.epochs} épocas)...")

result = loop.run(
    model=model,
    loss_fn=loss_fn,
    metrics_list=metrics_list,
    train_ds=train_ds,
    val_ds=val_ds,
    callbacks=callbacks,
)

print(f"\n✓ Treinamento concluído:")
print(f"  Melhor época   : {result.best_epoch}")
print(f"  Melhor val_loss: {result.best_val_loss:.6f}")
print(f"  Tempo total    : {result.training_time:.1f}s")
print(f"  Épocas treinadas: {len(result.history.get('loss', []))}")

## Célula 8 — Avaliação e Métricas

Avalia o modelo treinado no conjunto de validação e reporta métricas geofísicas:  
R² (separado por ρ_h e ρ_v), RMSE, MAE, AnisotropyRatioError.

In [ ]:
# ── Avaliação ────────────────────────────────────────────────────────────
# evaluate_predictions() computa métricas numpy sobre arrays numpy.
# Separado do model.evaluate() para maior flexibilidade.
# Assinatura: evaluate_predictions(y_true, y_pred, *, dataset_name="test")
# NÃO aceita config como parâmetro — usa keyword-only dataset_name.

from geosteering_ai.evaluation import evaluate_predictions

# Gera predições no conjunto de validação
y_pred_batches = []
for x_batch, _ in val_ds:
    y_pred_batch = model(x_batch, training=False)
    y_pred_batches.append(y_pred_batch.numpy())

y_pred_all = np.concatenate(y_pred_batches, axis=0)   # (N_val, 600, 2)
print(f"Predições: {y_pred_all.shape}")
print(f"Ground truth: {y_val.shape}")

# Avalia métricas
try:
    metrics_report = evaluate_predictions(y_val, y_pred_all, dataset_name="val")
    print(f"\n✓ Métricas de Avaliação:")
    print(f"  R² (ρ_h) : {metrics_report.r2_rh:.4f}")
    print(f"  R² (ρ_v) : {metrics_report.r2_rv:.4f}")
    print(f"  RMSE     : {metrics_report.rmse:.6f}")
    print(f"  MAE      : {metrics_report.mae:.6f}")
except Exception as e:
    # Fallback: métricas manuais caso evaluate_predictions retorne erro
    from sklearn.metrics import r2_score, mean_squared_error
    y_true_flat = y_val.reshape(-1, 2)
    y_pred_flat = y_pred_all.reshape(-1, 2)
    print(f"\n✓ Métricas (fallback sklearn):")
    print(f"  R² (ρ_h) : {r2_score(y_true_flat[:, 0], y_pred_flat[:, 0]):.4f}")
    print(f"  R² (ρ_v) : {r2_score(y_true_flat[:, 1], y_pred_flat[:, 1]):.4f}")
    print(f"  RMSE     : {float(np.sqrt(mean_squared_error(y_true_flat, y_pred_flat))):.6f}")

# Verifica ausência de NaN/Inf nas predições
assert not np.any(np.isnan(y_pred_all)), "NaN detectado nas predições!"
assert not np.any(np.isinf(y_pred_all)), "Inf detectado nas predições!"
print("\n✓ Predições livres de NaN/Inf")

## Célula 9 — Visualização dos Resultados

Plota amostras do holdout comparando predições vs. ground truth.  
4 componentes EM + 2 perfis de resistividade (ρ_h, ρ_v em escala log10).

In [ ]:
# ── Visualização ─────────────────────────────────────────────────────────
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline

matplotlib.rcParams['figure.dpi'] = 100

# Plota 1 amostra do holdout: predição vs. ground truth
sample_idx = 0
z_vals = x_val[sample_idx, :, 0]       # zobs
rho_h_true  = 10 ** y_val[sample_idx, :, 0]   # Inverter log10 → Ohm.m
rho_v_true  = 10 ** y_val[sample_idx, :, 1]
rho_h_pred  = 10 ** y_pred_all[sample_idx, :, 0]
rho_v_pred  = 10 ** y_pred_all[sample_idx, :, 1]

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# ρ_h
ax = axes[0]
ax.semilogx(rho_h_true, z_vals, 'k-', linewidth=2, label='Ground truth (ρ_h)')
ax.semilogx(rho_h_pred, z_vals, 'r--', linewidth=1.5, label='Predição (ρ_h)', alpha=0.8)
ax.set_xlabel('Resistividade Horizontal ρ_h [Ohm.m]')
ax.set_ylabel('Profundidade [m]')
ax.invert_yaxis()
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_title(f'ρ_h — Amostra {sample_idx}')

# ρ_v
ax = axes[1]
ax.semilogx(rho_v_true, z_vals, 'k-', linewidth=2, label='Ground truth (ρ_v)')
ax.semilogx(rho_v_pred, z_vals, 'b--', linewidth=1.5, label='Predição (ρ_v)', alpha=0.8)
ax.set_xlabel('Resistividade Vertical ρ_v [Ohm.m]')
ax.set_ylabel('Profundidade [m]')
ax.invert_yaxis()
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_title(f'ρ_v — Amostra {sample_idx}')

fig.suptitle(f'Inversão 1D — {active_config.model_type} (Smoke Test)\n'
             f'Dados sintéticos | {active_config.sequence_length} pts | '
             f'target_scaling={active_config.target_scaling}',
             fontsize=11)
plt.tight_layout()
plt.show()
plt.close()

# Plot do histórico de treinamento
if result.history and 'loss' in result.history:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(result.history['loss'], label='Train loss')
    if 'val_loss' in result.history:
        ax.plot(result.history['val_loss'], label='Val loss')
    ax.axvline(result.best_epoch, color='g', linestyle='--', alpha=0.7,
               label=f'Best epoch ({result.best_epoch})')
    ax.set_xlabel('Época')
    ax.set_ylabel('Loss')
    ax.set_title('Histórico de Treinamento')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    plt.close()

print("✓ Visualização concluída")

## Célula 10 — InferencePipeline: Salvar e Carregar

`InferencePipeline.save()` persiste: `model.keras` + `scalers.joblib` + `config.yaml`.  
`InferencePipeline.load()` restaura o estado completo para inferência.

In [ ]:
# ── InferencePipeline — Salvar e Carregar ────────────────────────────────
# InferencePipeline encapsula: modelo + scalers + config para inferência.
# Após save(), o pipeline pode ser carregado em qualquer ambiente.

import os
from geosteering_ai.inference import InferencePipeline

OUTPUT_DIR = "./outputs/smoke_test"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Cria InferencePipeline com modelo treinado
# scaler_params é None para smoke test (sem scalers reais)
inf_pipe = InferencePipeline(
    config=active_config,
    model=model,
    scaler_params=None,   # None para smoke test — sem scalers reais
)

# Salva o pipeline
try:
    inf_pipe.save(OUTPUT_DIR)
    print(f"✓ InferencePipeline salvo em: {OUTPUT_DIR}")
    print(f"  Arquivos gerados:")
    for f in os.listdir(OUTPUT_DIR):
        fpath = os.path.join(OUTPUT_DIR, f)
        size = os.path.getsize(fpath)
        print(f"    {f}: {size:,} bytes")
except RuntimeError as e:
    print(f"  Aviso: {e}")
    print("  (Normal para smoke test sem modelo compilado — continue para Célula 11)")

# Salva apenas o modelo Keras diretamente
model_path = os.path.join(OUTPUT_DIR, "model_smoke.keras")
model.save(model_path)
print(f"\n✓ Modelo salvo diretamente: {model_path}")
print(f"  Tamanho: {os.path.getsize(model_path):,} bytes")

## Célula 11 — Smoke Test GPU: Forward + Backward Pass

Valida explicitamente o comportamento em GPU:  
1. **Forward pass**: `model(X)` produz output no dispositivo correto  
2. **Backward pass**: `GradientTape` computa gradientes sem NaN/Inf  
3. **Latência**: mede tempo por batch (requisito: < 31 ms para geosteering RT)  
4. **Throughput**: amostras/segundo no hardware disponível  

In [ ]:
# ── Smoke Test GPU: Forward + Backward ──────────────────────────────────
# Valida o pipeline em GPU (ou CPU se GPU não disponível).
# Mede latência e throughput para avaliar viabilidade de uso em tempo real.

import time
import numpy as np
import tensorflow as tf

# Verifica dispositivo disponível
gpus = tf.config.list_physical_devices('GPU')
device_name = gpus[0].name if gpus else '/CPU:0'
print(f"Dispositivo ativo: {device_name}")
print(f"GPUs físicas: {len(gpus)}")

# Parâmetros do smoke test GPU
N_SMOKE_BATCHES   = 20      # Número de batches para aquecimento + medição
WARMUP_BATCHES    = 5       # Batches de aquecimento (descartados)
BATCH_SIZE_TEST   = 32      # Batch size do teste de latência
SEQ_LEN_TEST      = active_config.sequence_length   # 600
N_FEAT_TEST       = len(active_config.input_features)  # 5

# Gera batch de teste
x_test = tf.random.normal([BATCH_SIZE_TEST, SEQ_LEN_TEST, N_FEAT_TEST])
y_test_smoke = tf.random.normal([BATCH_SIZE_TEST, SEQ_LEN_TEST, 2])

print(f"\nBatch de teste: {x_test.shape} → esperado: ({BATCH_SIZE_TEST}, {SEQ_LEN_TEST}, 2)")

# ── 1. Forward Pass ──────────────────────────────────────────────────────
print("\n[1/4] Forward pass...")
_ = model(x_test, training=False)  # Aquecimento

latencies_fwd = []
for i in range(N_SMOKE_BATCHES):
    t0 = time.perf_counter()
    y_out = model(x_test, training=False)
    _ = y_out.numpy()  # Força sincronização GPU→CPU
    t1 = time.perf_counter()
    if i >= WARMUP_BATCHES:
        latencies_fwd.append((t1 - t0) * 1000)  # ms

lat_mean = np.mean(latencies_fwd)
lat_std  = np.std(latencies_fwd)
lat_p95  = np.percentile(latencies_fwd, 95)
throughput = BATCH_SIZE_TEST / (lat_mean / 1000)  # samples/s

print(f"  ✓ Forward pass OK")
print(f"  Latência média  : {lat_mean:.2f} ± {lat_std:.2f} ms")
print(f"  Latência p95    : {lat_p95:.2f} ms")
print(f"  Throughput      : {throughput:.0f} amostras/s")

# Requisito de geosteering RT: < 31 ms por batch de 32 amostras
RT_THRESHOLD_MS = 31.0
status = "✓ OK" if lat_mean < RT_THRESHOLD_MS else "⚠ ACIMA DO LIMIAR"
print(f"  Requisito RT    : < {RT_THRESHOLD_MS} ms → {status} ({lat_mean:.1f} ms)")

# ── 2. Backward Pass (GradientTape) ──────────────────────────────────────
print("\n[2/4] Backward pass (GradientTape)...")

latencies_bwd = []
for i in range(N_SMOKE_BATCHES):
    t0 = time.perf_counter()
    with tf.GradientTape() as tape:
        y_pred_tape = model(x_test, training=True)
        loss = loss_fn(y_test_smoke, y_pred_tape)
    grads = tape.gradient(loss, model.trainable_variables)
    _ = [g.numpy() if g is not None else None for g in grads[:3]]  # Sync parcial
    t1 = time.perf_counter()
    if i >= WARMUP_BATCHES:
        latencies_bwd.append((t1 - t0) * 1000)

lat_bwd_mean = np.mean(latencies_bwd)
print(f"  ✓ Backward pass OK")
print(f"  Latência fwd+bwd: {lat_bwd_mean:.2f} ms")

# ── 3. Verificar ausência de NaN/Inf ────────────────────────────────────
print("\n[3/4] Verificando NaN/Inf nos gradientes...")
n_valid = sum(1 for g in grads if g is not None)
n_nan   = sum(1 for g in grads if g is not None and tf.reduce_any(tf.math.is_nan(g)).numpy())
n_inf   = sum(1 for g in grads if g is not None and tf.reduce_any(tf.math.is_inf(g)).numpy())
print(f"  Gradientes válidos: {n_valid}/{len(model.trainable_variables)}")
print(f"  NaN: {n_nan} | Inf: {n_inf}")
assert n_nan == 0, f"ERRO: {n_nan} gradientes com NaN!"
assert n_inf == 0, f"ERRO: {n_inf} gradientes com Inf!"
print("  ✓ Nenhum NaN/Inf detectado")

# ── 4. Verificar Constraint TIV (ρ_v ≥ ρ_h) ────────────────────────────
print("\n[4/4] Verificando constraint TIV (ρ_v ≥ ρ_h na escala log10)...")
y_final = model(x_test, training=False).numpy()
rho_h_out = y_final[:, :, 0]   # log10(ρ_h)
rho_v_out = y_final[:, :, 1]   # log10(ρ_v)
tiv_violations = np.mean(rho_v_out < rho_h_out - 0.01)  # 1% tolerância
print(f"  Violações TIV (ρ_v < ρ_h): {tiv_violations*100:.2f}%")
if tiv_violations > 0.1:
    print("  ⚠ Muitas violações TIV — considere TIVConstraintLayer (Fase B)")
else:
    print("  ✓ TIV OK")

print(f"\n{'='*60}")
print(f"SMOKE TEST GPU CONCLUÍDO")
print(f"  Forward  : {lat_mean:.2f} ms/batch")
print(f"  Backward : {lat_bwd_mean:.2f} ms/batch")
print(f"  NaN/Inf  : 0")
print(f"  Params   : {model.count_params():,}")
print(f"{'='*60}")

## Célula 12 — Smoke Test: Integração com DataPipeline Real

Valida a integração com dados reais `.dat`/`.out` (quando disponíveis).  
Em modo sintético, valida o formato e a cadeia de transformações.

In [ ]:
# ── Smoke Test: Integração DataPipeline ─────────────────────────────────
# Valida a cadeia completa: .dat/.out → PreparedData → tf.data → model
# A cadeia deve ser fisicamente correta:
#   train_raw → noise(A/m) → FV_tf(noisy) → GS_tf(noisy) → scale → model

import dataclasses
import numpy as np
from geosteering_ai.data import DataPipeline
from geosteering_ai.noise import create_noise_level_var

print("Smoke Test: DataPipeline + Cadeia de Transformações")
print("-" * 50)

if USE_SYNTHETIC_DATA:
    # Valida as transformações individuais com dados sintéticos
    from geosteering_ai.data import apply_feature_view, VALID_VIEWS
    
    print("[Modo Sintético] Validando Feature Views...")
    
    # Dados sintéticos no formato 22-col
    rng = np.random.default_rng(123)
    n_pts = 600
    # Col layout: [zobs, rho_h, rho_v, Re(Hxx), Im(Hxx), ..., Re(Hzz), Im(Hzz)]
    data_22col = rng.standard_normal((n_pts, 22)).astype(np.float32)
    data_22col[:, 1] = rng.uniform(0.1, 1000, n_pts).astype(np.float32)  # ρ_h
    data_22col[:, 2] = data_22col[:, 1] * rng.uniform(1.0, 5.0, n_pts).astype(np.float32)  # ρ_v
    
    # Testa todas as 6 Feature Views
    test_config = PipelineConfig.baseline()
    errors = []
    
    for view in VALID_VIEWS:
        test_config_fv = dataclasses.replace(test_config, feature_view=view)
        try:
            result_fv = apply_feature_view(data_22col, test_config_fv)
            # Verifica shape: deve ter mesma n_rows e n_features correto
            assert result_fv.shape[0] == n_pts, f"Shape row mismatch: {result_fv.shape}"
            assert not np.any(np.isnan(result_fv)), f"NaN em Feature View '{view}'"
            print(f"  ✓ {view:30s}: shape={result_fv.shape}")
        except Exception as e:
            errors.append((view, str(e)))
            print(f"  ✗ {view:30s}: ERRO — {e}")
    
    if errors:
        print(f"\n⚠ {len(errors)} erros encontrados nas Feature Views!")
    else:
        print(f"\n✓ Todas as {len(VALID_VIEWS)} Feature Views funcionando corretamente")
    
    # Valida create_noise_level_var
    noise_var = create_noise_level_var(initial_value=0.05)
    print(f"\n✓ Noise level var: {float(noise_var):.4f} (dtype={noise_var.dtype})")
    
else:
    # Dados reais: executa DataPipeline.prepare() completo
    print(f"[Modo Real] Carregando {DAT_PATH}...")
    pipeline = DataPipeline(active_config)
    data = pipeline.prepare(DAT_PATH, OUT_PATH)
    
    # Cria map_fn para treinamento on-the-fly
    noise_var = create_noise_level_var(initial_value=0.0)
    map_fn = pipeline.build_train_map_fn(noise_var)
    
    # Cria tf.data datasets
    AUTOTUNE = tf.data.AUTOTUNE
    real_train_ds = (
        tf.data.Dataset.from_tensor_slices((data.x_train, data.y_train))
        .map(map_fn, num_parallel_calls=AUTOTUNE)
        .batch(active_config.batch_size)
        .prefetch(AUTOTUNE)
    )
    
    print(f"  ✓ Train: {data.x_train.shape} → {data.y_train.shape}")
    print(f"  ✓ Val  : {data.x_val.shape} → {data.y_val.shape}")
    print(f"  ✓ Test : {data.x_test.shape} → {data.y_test.shape}")
    
    # Smoke test: verifica 1 batch do dataset
    for x_batch, y_batch in real_train_ds.take(1):
        print(f"  ✓ Batch real: x={x_batch.shape}, y={y_batch.shape}")
        assert not tf.reduce_any(tf.math.is_nan(x_batch)), "NaN em x_batch!"
        assert not tf.reduce_any(tf.math.is_nan(y_batch)), "NaN em y_batch!"
        print("  ✓ Nenhum NaN no batch real")

print("\n✓ Smoke Test DataPipeline concluído")

## Célula 12B — Validação GPU End-to-End com Dados Reais (Inv0Dip13mil)

Validação completa do pipeline com o dataset real `Inv0Dip13mil.dat` no Google Colab Pro+ GPU.  
Esta célula executa: load → split → fit scaler → build model → train (5 epochs) → evaluate → report.

In [ ]:
# ── Validação GPU End-to-End com Dados Reais ─────────────────────────────
# Pipeline completo: Inv0Dip13mil.dat → DataPipeline → Model → Train → Eval
# Só executa no Colab com acesso ao Google Drive.
#
# Dataset: Inv0Dip13mil (13.000 modelos geológicos 1D, 22 colunas)
#   Train: ~10.400 modelos (70%), Val: ~1.300 (15%), Test: ~1.300 (15%)
#   Cada modelo: 600 medidas × 22 colunas → (600, 5) features + (600, 2) targets
#
# Validações:
#   1. DataPipeline.prepare() carrega e splita corretamente
#   2. Scaler fitado em dados LIMPOS (não ruidosos)
#   3. Forward pass: shapes (batch, 600, 2) preservados
#   4. Backward pass: gradients sem NaN/Inf
#   5. Físico: ρ_h, ρ_v em faixa esperada [0.01, 100000] Ohm.m
#   6. GPU: timing (ms/batch) e memória (MB)

import os
import time
import logging

logger = logging.getLogger(__name__)

# ── Verificação de ambiente ──────────────────────────────────────────────
try:
    import google.colab  # noqa: F401
    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

# Paths dos datasets reais (Google Drive Colab Pro+)
_DRIVE_BASE = "/content/drive/MyDrive/DataSets/Resistivity_Projeto_Pos_Doutorado"
_REAL_DAT_TRAIN   = os.path.join(_DRIVE_BASE, "Inv0Dip13mil.dat")
_REAL_OUT_TRAIN   = os.path.join(_DRIVE_BASE, "Inv0Dip13mil.out")
_REAL_DAT_HOLDOUT = os.path.join(_DRIVE_BASE, "Inv0Dip13mil_holdout.dat")
_REAL_OUT_HOLDOUT = os.path.join(_DRIVE_BASE, "Inv0Dip13mil_holdout.out")

_HAS_REAL_DATA = os.path.exists(_REAL_DAT_TRAIN) and os.path.exists(_REAL_OUT_TRAIN)

if not _IN_COLAB:
    print("⏭ SKIP: Não está executando no Google Colab.")
    print("  Esta célula requer Colab Pro+ com Google Drive montado.")
elif not _HAS_REAL_DATA:
    print("⏭ SKIP: Dados reais Inv0Dip13mil não encontrados.")
    print(f"  Esperado: {_REAL_DAT_TRAIN}")
    print("  Monte o Google Drive: drive.mount('/content/drive')")
else:
    import numpy as np
    import tensorflow as tf
    import dataclasses
    from geosteering_ai.config import PipelineConfig
    from geosteering_ai.data import DataPipeline
    from geosteering_ai.models import build_model
    from geosteering_ai.losses import get_loss
    from geosteering_ai.training import build_callbacks
    from geosteering_ai.noise import create_noise_level_var
    from geosteering_ai.evaluation import evaluate_predictions
    from geosteering_ai.utils import setup_gpu, gpu_memory_info

    print("=" * 60)
    print("VALIDAÇÃO GPU END-TO-END — Inv0Dip13mil (Dados Reais)")
    print("=" * 60)

    # ── 1. Config para validação rápida (5 épocas) ──────────────────────
    _gpu_config = dataclasses.replace(
        PipelineConfig.robusto(),
        epochs=5,
        early_stopping_patience=5,
        batch_size=32,
        use_noise=True,
        use_curriculum=False,      # Desativa curriculum para teste rápido
        noise_level_max=0.04,      # Nível fixo para validação
        use_tensorboard=False,
        use_csv_logger=False,
        experiment_tag="gpu_validation_inv0dip13mil",
    )
    print(f"\n[1/6] Config: {_gpu_config.model_type}, {_gpu_config.epochs} épocas")

    # ── 2. DataPipeline: load → split → scaler ──────────────────────────
    t0 = time.perf_counter()
    _gpu_pipeline = DataPipeline(_gpu_config)
    _gpu_data = _gpu_pipeline.prepare(_REAL_DAT_TRAIN, _REAL_OUT_TRAIN)
    t_load = time.perf_counter() - t0

    print(f"\n[2/6] DataPipeline ({t_load:.1f}s):")
    print(f"  Train: {data.x_train.shape} → {data.y_train.shape}")
    print(f"  Val  : {data.x_val.shape} → {data.y_val.shape}")
    print(f"  Test : {data.x_test.shape} → {data.y_test.shape}")
    print(f"  Modelos: train={data.n_models_train}, val={data.n_models_val}")

    # Validação: split por modelo, não por amostra
    assert _gpu_data.x_train.shape[1] == _gpu_config.sequence_length, "Erro no sequence_length!"
    assert _gpu_data.x_train.shape[2] == len(_gpu_config.input_features), "Erro em n_features!"
    assert _gpu_data.y_train.shape[2] == len(_gpu_config.output_targets), "Erro em output_channels!"

    # ── 3. Build model ──────────────────────────────────────────────────
    _gpu_model = build_model(_gpu_config)
    _gpu_n_params = _gpu_model.count_params()
    print(f"\n[3/6] Modelo: {_gpu_config.model_type} ({_gpu_n_params:,} params)")

    # ── 4. Compilar e treinar ────────────────────────────────────────────
    _gpu_loss_fn = get_loss(_gpu_config)
    _gpu_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=_gpu_config.learning_rate),
        loss=_gpu_loss_fn,
    )

    _gpu_noise_var = create_noise_level_var(initial_value=_gpu_config.noise_level_max)
    _gpu_map_fn = _gpu_pipeline.build_train_map_fn(_gpu_noise_var)

    AUTOTUNE = tf.data.AUTOTUNE
    _gpu_train_ds = (
        tf.data.Dataset.from_tensor_slices((_gpu_data.x_train, _gpu_data.y_train))
        .map(_gpu_map_fn, num_parallel_calls=AUTOTUNE)
        .batch(_gpu_config.batch_size)
        .prefetch(AUTOTUNE)
    )
    _gpu_val_ds = (
        tf.data.Dataset.from_tensor_slices((_gpu_data.x_val, _gpu_data.y_val))
        .batch(_gpu_config.batch_size)
        .prefetch(AUTOTUNE)
    )

    t0 = time.perf_counter()
    _gpu_history = _gpu_model.fit(
        _gpu_train_ds,
        validation_data=_gpu_val_ds,
        epochs=_gpu_config.epochs,
        verbose=1,
    )
    t_train = time.perf_counter() - t0

    print(f"\n[4/6] Treinamento ({t_train:.1f}s, {t_train/_gpu_config.epochs:.1f}s/época):")
    final_loss = _gpu_history.history['loss'][-1]
    final_val_loss = _gpu_history.history['val_loss'][-1]
    print(f"  Loss final      : {final_loss:.6f}")
    print(f"  Val loss final  : {final_val_loss:.6f}")

    # ── 5. Avaliação e validação física ──────────────────────────────────
    _gpu_y_pred = _gpu_model.predict(_gpu_data.x_test, verbose=0)
    assert _gpu_y_pred.shape == _gpu_data.y_test.shape, f"Shape mismatch: {_gpu_y_pred.shape} vs {_gpu_data.y_test.shape}"

    # Validação física: targets são log10(rho), rho em [0.01, 100000]
    # Portanto log10(rho) em [-2, 5]
    _gpu_y_test_np = _gpu_data.y_test
    pred_min = np.min(_gpu_y_pred)
    pred_max = np.max(_gpu_y_pred)
    true_min = np.min(_gpu_y_test_np)
    true_max = np.max(_gpu_y_test_np)

    print(f"\n[5/6] Validação Física:")
    print(f"  y_pred range : [{pred_min:.3f}, {pred_max:.3f}] (log10 Ohm.m)")
    print(f"  y_true range : [{true_min:.3f}, {true_max:.3f}] (log10 Ohm.m)")
    print(f"  NaN em y_pred: {np.any(np.isnan(_gpu_y_pred))}")
    print(f"  Inf em y_pred: {np.any(np.isinf(_gpu_y_pred))}")

    _gpu_metrics = evaluate_predictions(_gpu_y_test_np, _gpu_y_pred, dataset_name="Inv0Dip13mil_holdout")
    for k, v in _gpu_metrics.items():
        print(f"  {k}: {v:.6f}")

    # ── 6. GPU timing ────────────────────────────────────────────────────
    gpus = tf.config.list_physical_devices('GPU')
    _gpu_mem_info = gpu_memory_info() if gpus else {}

    # Inferência timing (100 batches)
    _gpu_test_batch = _gpu_data.x_test[:_gpu_config.batch_size]
    _ = _gpu_model(_gpu_test_batch, training=False)  # warmup
    t0 = time.perf_counter()
    for _ in range(100):
        _ = _gpu_model(_gpu_test_batch, training=False)
    t_infer = (time.perf_counter() - t0) / 100 * 1000  # ms

    print(f"\n[6/6] GPU Performance:")
    print(f"  Dispositivo      : {gpus[0].name if gpus else 'CPU'}")
    print(f"  GPU memory (MB)  : {_gpu_mem_info.get('used_mb', 'N/A')}")
    print(f"  Inferência       : {t_infer:.2f} ms/batch ({_gpu_config.batch_size} amostras)")
    print(f"  Treinamento      : {t_train/_gpu_config.epochs:.1f}s/época")

    print("\n" + "=" * 60)
    print("✓ VALIDAÇÃO GPU END-TO-END COMPLETA — Inv0Dip13mil")
    print("=" * 60)

## Célula 13 — Smoke Test: Modo Realtime (Geosteering Causal)

Valida o pipeline de inferência em tempo real:  
- `RealtimeInference` com sliding window (deque, maxlen=600)  
- Padding causal (sem dados futuros)  
- Latência < 10 ms por amostra (requisito geosteering)

In [ ]:
# ── Smoke Test: Inferência Realtime ──────────────────────────────────────
# Valida o modo realtime com sliding window.
# RealtimeInference requer um InferencePipeline previamente construído.
# Usa deque(maxlen=600) para manter apenas os últimos 600 pontos de medição.
# Assinatura: RealtimeInference(pipeline: InferencePipeline, window_size=600)

import time
from geosteering_ai.inference import InferencePipeline, RealtimeInference

# Nota: Para realtime real, use model_type causal (WaveNet, Causal_Transformer, etc.)
# Aqui usamos o modelo treinado para smoke test apenas

try:
    # Constrói InferencePipeline primeiro (necessário para RealtimeInference)
    inf_pipe_rt = InferencePipeline(
        config=active_config,
        model=model,
        scaler_params=None,   # None para smoke test — sem scalers reais
    )

    # Constrói RealtimeInference com o pipeline
    rt_inf = RealtimeInference(
        pipeline=inf_pipe_rt,
        window_size=active_config.sequence_length,
    )

    # Simula 50 medições chegando sequencialmente
    N_RT_SAMPLES = 50
    latencies_rt = []

    n_feat_rt = len(active_config.input_features)

    for i in range(N_RT_SAMPLES):
        # Simula chegada de 1 nova medição (vetor de n_feat_rt valores)
        new_sample = np.random.randn(n_feat_rt).astype(np.float32)
        new_sample[0] = float(i)  # zobs = profundidade atual

        t0 = time.perf_counter()
        prediction = rt_inf.update(new_sample)
        t1 = time.perf_counter()
        latencies_rt.append((t1 - t0) * 1000)

    lat_rt_mean = np.mean(latencies_rt[5:])   # Skip warm-up
    lat_rt_p95  = np.percentile(latencies_rt[5:], 95)

    print(f"✓ RealtimeInference OK")
    print(f"  Amostras processadas: {N_RT_SAMPLES}")
    print(f"  Latência média : {lat_rt_mean:.2f} ms/amostra")
    print(f"  Latência p95   : {lat_rt_p95:.2f} ms/amostra")

    RT_SAMPLE_THRESHOLD_MS = 10.0
    status_rt = "✓ OK" if lat_rt_mean < RT_SAMPLE_THRESHOLD_MS else "⚠ ACIMA"
    print(f"  Requisito RT   : < {RT_SAMPLE_THRESHOLD_MS} ms → {status_rt} ({lat_rt_mean:.1f} ms)")

    if prediction is not None:
        print(f"  Output formato : {prediction.shape} (últimas predições)")

except Exception as e:
    print(f"  ⚠ RealtimeInference smoke test parcial: {e}")
    print("  Validando interface básica (instanciação)...")
    inf_pipe_rt = InferencePipeline(config=active_config, model=model, scaler_params=None)
    rt_inf = RealtimeInference(pipeline=inf_pipe_rt)
    print(f"  ✓ RealtimeInference instanciado: buffer_size={rt_inf.window_size}")

print("\n✓ Smoke Test Realtime concluído")

## Célula 14 — Relatório Final do Smoke Test

Consolida todos os resultados dos smoke tests e gera um relatório de status.

In [ ]:
# ── Relatório Final do Smoke Test ────────────────────────────────────────
import json
from datetime import datetime

report = {
    "timestamp"       : datetime.now().isoformat(),
    "geosteering_ai_version": geosteering_ai.__version__,
    "tensorflow_version"    : tf.__version__,
    "hardware": {
        "gpu_count"    : gpu_info['gpu_count'],
        "device"       : device_name,
    },
    "model": {
        "type"         : active_config.model_type,
        "n_params"     : model.count_params(),
        "sequence_length": active_config.sequence_length,
        "n_features"   : len(active_config.input_features),
        "output_channels": 2,
    },
    "training": {
        "epochs_run"   : len(result.history.get('loss', [])),
        "best_epoch"   : result.best_epoch,
        "best_val_loss": result.best_val_loss,
        "training_time_s": result.training_time,
    },
    "performance": {
        "forward_lat_ms"   : round(lat_mean, 2),
        "forward_lat_p95_ms": round(lat_p95, 2),
        "fwd_bwd_lat_ms"   : round(lat_bwd_mean, 2),
        "throughput_sps"   : round(throughput, 0),
        "rt_requirement_ok": lat_mean < 31.0,
    },
    "correctness": {
        "nan_gradients"  : n_nan,
        "inf_gradients"  : n_inf,
        "tiv_violations_pct": round(tiv_violations * 100, 2),
    },
}

# Salva relatório JSON
report_path = "./outputs/smoke_test/smoke_test_report.json"
os.makedirs(os.path.dirname(report_path), exist_ok=True)
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print("=" * 65)
print("RELATÓRIO FINAL — GEOSTEERING AI v2.0 SMOKE TEST")
print("=" * 65)
print(f"Timestamp    : {report['timestamp'][:19]}")
print(f"Versão       : geosteering_ai v{report['geosteering_ai_version']}")
print(f"TensorFlow   : {report['tensorflow_version']}")
print(f"Dispositivo  : {device_name}")
print()
print(f"Modelo       : {active_config.model_type}")
print(f"Parâmetros   : {model.count_params():,}")
print()
print(f"Treinamento  : {len(result.history.get('loss', []))} épocas")
print(f"Melhor epoch : {result.best_epoch}")
print(f"Best val_loss: {result.best_val_loss:.6f}")
print()
print(f"Forward lat  : {lat_mean:.2f} ± {lat_std:.2f} ms  (p95={lat_p95:.2f}ms)")
print(f"Throughput   : {throughput:.0f} amostras/s")
print(f"RT requisito : {'✓ OK' if lat_mean < 31.0 else '⚠ FALHOU'} (< 31 ms)")
print()
print(f"NaN grads    : {n_nan}")
print(f"Inf grads    : {n_inf}")
print(f"TIV violações: {tiv_violations*100:.2f}%")
print()
print(f"Status       : {'✓ PASSOU' if n_nan == 0 and n_inf == 0 else '✗ FALHOU'}")
print(f"Relatório    : {report_path}")
print("=" * 65)